# Predicting Crime Hotspots in Bexley Using Metropolitan Police Data

## Project Overview
This project investigates crime patterns in the London Borough of Bexley using Metropolitan Police data and applies machine learning to identify areas that are most likely to become crime hotspots. The analysis begins by cleaning and exploring the dataset, examining crime types, time trends, and geographic concentration across Bexley LSOAs. The project then transforms the original incident-level data into an area-month hotspot prediction task, allowing crime risk to be modelled in a more realistic and interpretable way. Visualisations — including charts and interactive maps — highlight where crime is concentrated, while classification models (including Random Forest) are trained and evaluated to predict hotspot risk.

## Project Workflow
1. **Problem Framing & Data Acquisition**
2. **Exploratory Data Analysis (EDA) & Data Preparation**
3. **Model Development, Evaluation & Business Interpretation**


## Setup & Imports


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn — preprocessing & evaluation
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


---
# Deliverable 1: Problem Framing & Data Acquisition

## Problem Statement
The aim of this project is to predict which areas of the London Borough of **Bexley** are most likely to become **crime hotspots** in a given month using Metropolitan Police incident data. Rather than predicting an exact street point, the project focuses on the `LSOA` level — a more realistic and useful spatial unit for public safety analysis.

## Target Variable
**`Hotspot`** — a binary feature engineered during data preparation. An `LSOA-month` is labelled as a hotspot when its crime count falls in the upper quartile (≥ 75th percentile) of monthly crime volume across the Bexley dataset.

## Dataset Source
Monthly **Metropolitan Police** CSV files covering **March 2023 to February 2026**, filtered to Bexley `street`-level records. These contain the `Crime type`, `Location`, `Longitude`, `Latitude`, `LSOA code`, and `LSOA name` fields needed for mapping and hotspot engineering.

## Data Acquisition Summary
1. Collect monthly Metropolitan Police CSV files for `2023-03` to `2026-02`.
2. Merge all monthly files into one combined dataset for inspection.
3. Filter to Bexley rows only using the `LSOA name` field.
4. Retain only `street` schema records (outcome records lack the spatial columns required).
5. Save the filtered file as **`combined_metropolitan_data_bexley_street_only.csv`**.


## Data Loading

The notebook expects `combined_metropolitan_data_bexley_street_only.csv` in the working directory. If running on Google Colab, upload the file when prompted.


In [ ]:
from pathlib import Path

data_path = Path('combined_metropolitan_data_bexley_street_only.csv')

if not data_path.exists():
    try:
        from google.colab import files
        print('Please upload combined_metropolitan_data_bexley_street_only.csv')
        files.upload()
    except ImportError:
        print('Dataset not found. Place the CSV in the same directory as this notebook.')

df = pd.read_csv(data_path)
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head()


---
# Deliverable 2: Exploratory Data Analysis & Data Preparation

This section explores the Bexley street-level crime data and prepares it for modelling. The notebook examines data quality and spatial patterns, then engineers a monthly hotspot target at the `LSOA name` level so the model can answer: **which Bexley areas are most likely to become crime hotspots in a given month?**


## Data Inspection


In [ ]:
print('Shape:', df.shape)
df.info()


In [ ]:
# Missing value audit
missing_summary = df.isnull().sum().sort_values(ascending=False).to_frame('Missing Values')
missing_summary['Missing %'] = (missing_summary['Missing Values'] / len(df) * 100).round(2)
display(missing_summary)

# Drop columns that are entirely null or provide no predictive value
# - 'Outcome type' / 'Context': 100% null in the street-filtered dataset
# - 'Crime ID': identifier only
# - 'Last outcome category': missing values and post-event leakage risk
columns_to_drop = ['Outcome type', 'Context', 'Crime ID', 'Last outcome category']
df = df.drop(columns=columns_to_drop)
print('Remaining columns:', df.columns.tolist())


## Feature Engineering & Hotspot Target

Incidents are aggregated to an `LSOA name × Year × MonthNum` grain. An LSOA-month is labelled `Hotspot = 1` if its crime count meets or exceeds the **75th percentile** of all monthly crime counts across Bexley — matching the threshold used in the UI.


In [ ]:
df['Month'] = pd.to_datetime(df['Month'])
df['Year'] = df['Month'].dt.year
df['MonthNum'] = df['Month'].dt.month

# Aggregate to LSOA-month grain
monthly_hotspots = df.groupby(['LSOA name', 'Year', 'MonthNum']).size().reset_index(name='CrimeCount')
hotspot_threshold = monthly_hotspots['CrimeCount'].quantile(0.75)
monthly_hotspots['Hotspot'] = (monthly_hotspots['CrimeCount'] >= hotspot_threshold).astype(int)

print(f'Hotspot threshold (75th percentile): {hotspot_threshold}')
print('\nHotspot class balance:')
print(monthly_hotspots['Hotspot'].value_counts(normalize=True).round(3))
monthly_hotspots.head()


## Visualisations

The charts below summarise crime type distribution, monthly incident trends, and the Bexley areas with the highest recorded crime counts.


In [ ]:
# Crime type distribution and monthly trend
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

crime_counts = df['Crime type'].value_counts().head(10)
sns.barplot(x=crime_counts.values, y=crime_counts.index, ax=axes[0], palette='Blues_r')
axes[0].set_title('Top 10 Crime Types in Bexley')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Crime type')

monthly_counts = df.groupby(df['Month'].dt.to_period('M')).size()
monthly_counts.index = monthly_counts.index.astype(str)
sns.lineplot(x=monthly_counts.index, y=monthly_counts.values, marker='o', ax=axes[1], color='darkred')
axes[1].set_title('Monthly Crime Incidents in Bexley')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Incident count')
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()


In [ ]:
# LSOAs with the highest total crime counts
top_lsoa_counts = df['LSOA name'].value_counts().head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_lsoa_counts.values, y=top_lsoa_counts.index, palette='viridis')
plt.title('Top 10 Bexley LSOAs by Crime Count')
plt.xlabel('Count')
plt.ylabel('LSOA name')
plt.tight_layout()
plt.show()


In [ ]:
!pip install folium branca -q

import folium
from folium.plugins import HeatMap
import branca.colormap as cm

# Sample for performance if the dataset is large
map_df = df[['Latitude', 'Longitude']].dropna()
if len(map_df) > 10000:
    map_df = map_df.sample(10000, random_state=42)

crime_map = folium.Map(
    location=[map_df['Latitude'].mean(), map_df['Longitude'].mean()],
    zoom_start=11,
    tiles='CartoDB positron'
)
HeatMap(map_df[['Latitude', 'Longitude']].values.tolist(), radius=8, blur=6).add_to(crime_map)
crime_map


## Data Preparation Summary

For the spatial prediction task the incident-level data is transformed into an `LSOA-month` dataset. Key decisions:

- `Outcome type` and `Context` dropped — 100% null in the Bexley street dataset.
- `Crime ID` dropped — identifier only, no predictive value.
- `Last outcome category` dropped — contains missing values and reflects post-event information that could introduce leakage.
- `Year` and `MonthNum` extracted from the `Month` column.
- Incidents aggregated to `LSOA name × Year × MonthNum` grain.
- Binary `Hotspot` target defined using the 75th-percentile of monthly crime counts.

`Month`, `LSOA code`, `LSOA name`, `Crime type`, `Longitude`, and `Latitude` contain no missing values in the filtered dataset.


---
# Deliverable 3: Model Development, Evaluation & Interpretation

Multiple classification models are compared for predicting whether a Bexley LSOA will be a crime hotspot in a given month. Random Forest is the primary model and is subsequently tuned with grid search.


## Modeling Dataset Setup


In [ ]:
# Features: LSOA name (area), Year, MonthNum (time)
# Target: Hotspot (1 = ≥ 75th percentile monthly crime count)
model_df = monthly_hotspots.copy()
X = model_df[['LSOA name', 'Year', 'MonthNum']]
y = model_df['Hotspot']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training rows: {len(X_train)}')
print(f'Test rows:     {len(X_test)}')
print('\nTarget distribution:')
print(y.value_counts(normalize=True).round(3))


In [ ]:
# Separate categorical and numeric features
categorical_features = ['LSOA name']
numeric_features = ['Year', 'MonthNum']

categorical_features, numeric_features


In [ ]:
# Preprocessing pipeline
# - LSOA name: impute (most frequent) → one-hot encode
# - Year / MonthNum: impute (median)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), numeric_features)
    ]
)

preprocessor


## Model Comparison


In [ ]:
def evaluate_model(name, pipeline, X_train, X_test, y_train, y_test):
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall': recall_score(y_test, preds, zero_division=0),
        'F1': f1_score(y_test, preds, zero_division=0),
        'Kappa': cohen_kappa_score(y_test, preds)
    }

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=8),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=12, class_weight='balanced',
        random_state=42, n_jobs=1
    )
}

results = []
fitted_models = {}
for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    results.append(evaluate_model(name, pipe, X_train, X_test, y_train, y_test))
    fitted_models[name] = pipe.fit(X_train, y_train)

results_df = pd.DataFrame(results).sort_values('F1', ascending=False)
results_df


In [ ]:
# Baseline Random Forest confusion matrix
rf_baseline = fitted_models['Random Forest']
rf_preds = rf_baseline.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, rf_preds, cmap='Blues')
plt.title('Baseline Random Forest — Confusion Matrix')
plt.show()


## Hyperparameter Tuning

A grid search (3-fold CV, scoring on F1) tunes the Random Forest over `n_estimators`, `max_depth`, `min_samples_leaf`, and `class_weight`. `n_jobs=1` is used for compatibility in constrained environments.


In [ ]:
rf_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestClassifier(random_state=42, n_jobs=1))
])

param_grid = {
    'model__n_estimators': [200, 400],
    'model__max_depth': [12, None],
    'model__min_samples_leaf': [1, 2],
    'model__class_weight': ['balanced', 'balanced_subsample']
}

grid_search = GridSearchCV(rf_pipeline, param_grid=param_grid, cv=3, scoring='f1', n_jobs=1)
grid_search.fit(X_train, y_train)
print('Best parameters:')
grid_search.best_params_


In [ ]:
# Final evaluation on the held-out test set
best_rf = grid_search.best_estimator_
best_preds = best_rf.predict(X_test)

print(classification_report(y_test, best_preds, zero_division=0))

final_metrics = pd.DataFrame([{
    'Accuracy': accuracy_score(y_test, best_preds),
    'Precision': precision_score(y_test, best_preds, zero_division=0),
    'Recall': recall_score(y_test, best_preds, zero_division=0),
    'F1': f1_score(y_test, best_preds, zero_division=0),
    'Kappa': cohen_kappa_score(y_test, best_preds)
}])
final_metrics


In [ ]:
# Tuned model confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, best_preds, cmap='Greens')
plt.title('Tuned Random Forest — Confusion Matrix')
plt.show()


## Feature Importance

Feature importances from the tuned Random Forest show the relative contribution of each variable. Because `LSOA name` is one-hot encoded, its influence is spread across many binary columns — the UI's composite score weights (60% LSOA / 25% month / 15% year) reflect the aggregate importance ordering seen here.


In [ ]:
feature_names = best_rf.named_steps['prep'].get_feature_names_out()
importances = best_rf.named_steps['model'].feature_importances_
feature_importance_df = (
    pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    .sort_values('Importance', ascending=False)
)
feature_importance_df.head(15)


In [ ]:
# Visualise the top feature importances
top_features = feature_importance_df.head(12).sort_values('Importance')
plt.figure(figsize=(10, 6))
plt.barh(top_features['Feature'], top_features['Importance'], color='teal')
plt.title('Top 12 Random Forest Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## Results & Interpretation

The hotspot model should be read as a **borough-level risk signal**, not a street-level prediction tool. Key interpretive points:

- **Area dominates**: `LSOA name` carries the most predictive weight, confirming that some Bexley areas show persistent crime concentration regardless of the month or year.
- **Seasonality matters**: `MonthNum` adds meaningful signal; the summer-peak pattern visible in the trend chart is captured by the model.
- **Precision vs recall**: In a public safety context, recall (catching most real hotspots) is generally more important than precision. Review the classification report above to assess the trade-off.
- **Year effect is modest**: The `Year` feature contributes the least importance, suggesting no dramatic multi-year shift in Bexley crime concentration patterns within this dataset window.


## LSOA Crime Concentration


In [ ]:
# Top LSOAs by total crime count
lsoa_summary = (
    df.groupby(['LSOA code', 'LSOA name'])
      .size()
      .reset_index(name='CrimeCount')
      .sort_values('CrimeCount', ascending=False)
)
display(lsoa_summary.head(15))


In [ ]:
# Top LSOAs by hotspot rate (how consistently each area is a hotspot over time)
# NOTE: this hotspot_rank DataFrame is referenced by the UI for calibrating LSOA risk scores.
monthly_hotspots_with_code = (
    df.groupby(['LSOA code', 'LSOA name', 'Year', 'MonthNum'])
      .size()
      .reset_index(name='CrimeCount')
)
monthly_hotspots_with_code['Hotspot'] = (
    monthly_hotspots_with_code['CrimeCount'] >= hotspot_threshold
).astype(int)

hotspot_rank = (
    monthly_hotspots_with_code
    .groupby(['LSOA code', 'LSOA name'])['Hotspot']
    .mean()
    .reset_index(name='HotspotRate')
    .sort_values('HotspotRate', ascending=False)
)

# Safety score: round((1 - HotspotRate) * 100)
# Use this column to replace approximated risk scores in the UI.
hotspot_rank['SafetyScore'] = (hotspot_rank['HotspotRate'].apply(lambda x: round((1 - x) * 100)))
display(hotspot_rank.head(15))


In [ ]:
# Interactive LSOA circle map — size and colour scaled by crime count
lsoa_map_data = (
    df.groupby(['LSOA code', 'LSOA name'])
      .agg(Latitude=('Latitude', 'mean'), Longitude=('Longitude', 'mean'))
      .reset_index()
      .merge(lsoa_summary, on=['LSOA code', 'LSOA name'])
)

map_data = lsoa_map_data.head(15).copy()

lsoa_circle_map = folium.Map(
    location=[df['Latitude'].mean(), df['Longitude'].mean()],
    zoom_start=11,
    tiles='CartoDB positron'
)

colormap = cm.linear.YlOrRd_09.scale(
    map_data['CrimeCount'].min(), map_data['CrimeCount'].max()
)
colormap.caption = 'Crime Count by LSOA'
colormap.add_to(lsoa_circle_map)

for _, row in map_data.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=6 + (row['CrimeCount'] / map_data['CrimeCount'].max()) * 12,
        popup=(
            f"LSOA code: {row['LSOA code']}<br>"
            f"LSOA name: {row['LSOA name']}<br>"
            f"Crime count: {row['CrimeCount']}"
        ),
        color=colormap(row['CrimeCount']),
        fill=True,
        fill_color=colormap(row['CrimeCount']),
        fill_opacity=0.8
    ).add_to(lsoa_circle_map)

lsoa_circle_map


---
## Conclusion

The analysis showed that crime in Bexley is concentrated in a relatively small number of LSOA areas rather than being evenly distributed across the borough. The visualisations highlighted both the areas with the highest total crime counts and those that most consistently behave as hotspots over time. By transforming the original incident-level data into an LSOA-month hotspot dataset, the project created a realistic and defensible prediction task. The final tuned Random Forest demonstrated that area and time variables carry useful predictive information — supporting the conclusion that machine learning can assist borough-level crime analysis and help identify areas at elevated risk in a given month.


---
# Optional: Export Project Artifacts

The cells below export the engineered hotspot dataset and the tuned Random Forest model for reuse.

> **Tip for the UI:** replace the approximated `risk` values in `bexley_safety_ui.html` with the `SafetyScore` column from `hotspot_rank` above for a more accurate map.


In [ ]:
import json

# Build a short-name → SafetyScore mapping for the UI
# Keys match the ONS LSOA name prefix used in bexley_safety_ui.html
# e.g. 'Bexley 001A' matches the HTML entry 'Bexley 001A - Erith North'
risk_map = dict(zip(hotspot_rank['LSOA name'], hotspot_rank['SafetyScore']))

with open('lsoa_risk.json', 'w') as f:
    json.dump(risk_map, f, indent=2)

print(f'Exported {len(risk_map)} LSOA risk scores to lsoa_risk.json')
print('Upload this file alongside bexley_safety_ui.html in your GitHub repo.')
print()
print('Preview:')
for name, score in list(risk_map.items())[:5]:
    print(f'  {name}: {score}')


In [ ]:
import joblib

monthly_hotspots.to_csv('bexley_monthly_hotspots.csv', index=False)
hotspot_rank.to_csv('bexley_hotspot_rank.csv', index=False)
joblib.dump(best_rf, 'bexley_hotspot_random_forest.joblib')

print('Saved:')
print('  bexley_monthly_hotspots.csv')
print('  bexley_hotspot_rank.csv')
print('  bexley_hotspot_random_forest.joblib')
print('  lsoa_risk.json  (produced by the cell above)')
